# Paired t-test on differences: fusion vs image-only (one-sided)

In [ ]:
import json
import os
import re
import statistics
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

from scipy.stats import ttest_1samp


def load_test_macro_f1(exp_name: str) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        out[int(m.group(1))] = float(obj["test_metrics"]["test_macro_f1"])
    return out


fusion = load_test_macro_f1("phase3_robustness")
image = load_test_macro_f1("imageonly_robustness")
seeds = sorted(fusion.keys() & image.keys())
if fusion.keys() != image.keys():
    raise ValueError(f"seed mismatch: only fusion {sorted(set(fusion)-set(image))} only image {sorted(set(image)-set(fusion))}")

d = [fusion[s] - image[s] for s in seeds]
mean_d = statistics.mean(d)
std_d = statistics.stdev(d) if len(d) > 1 else float("nan")
cohen_d = mean_d / std_d if std_d else float("nan")
wins = sum(1 for x in d if x > 0)
ties = sum(1 for x in d if x == 0)
losses = sum(1 for x in d if x < 0)

res = ttest_1samp(d, popmean=0.0, alternative="greater")

alpha = 0.05
print("n", len(d))
print("mean_diff", mean_d)
print("stdev_diff", std_d)
print("paired_Cohen_d", cohen_d)
print("wins_ties_losses", wins, ties, losses)
print("median_diff", statistics.median(d))
print("statistic", res.statistic)
print("pvalue", res.pvalue)
print("df", len(d) - 1)
print("reject_H0_fusion_gt_image", res.pvalue < alpha)